# Phase 1: OCR bake-off (12 test pages)

Runs **PaddleOCR-VL 1.6** and **olmOCR-2 (7B)** on the 12 hard test pages and saves each page's text as Markdown.

**Before Run All:** Settings → Accelerator = **GPU T4 x2**, Internet = **On**, and **Add Input** → your private dataset
holding `phase1_pages.pdf`.

**After:** Output tab → download `phase1_results.zip` → put it in `agentic_rag_medical/data/phase1/`.

Each model runs in its own Python process, so a crash in one doesn't stop the other. Expected time: ~30–50 min.

In [ ]:
# 1. Setup: find the PDF, show the GPUs, install uv + PyMuPDF, render the pages
import glob, json, os, subprocess, time
PDF = (glob.glob("/kaggle/input/**/phase1_pages.pdf", recursive=True) or [None])[0]
assert PDF, "phase1_pages.pdf not found: Add Input -> your dataset"
OUT = "/kaggle/working/phase1"
os.makedirs(OUT, exist_ok=True)
print("PDF:", PDF)
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null
UV = os.path.expanduser("~/.local/bin/uv")
!{UV} pip install --system -q pymupdf
import pymupdf
doc = pymupdf.open(PDF)
os.makedirs(f"{OUT}/png200", exist_ok=True)
for i, page in enumerate(doc, start=1):            # 200 dpi pictures for PaddleOCR-VL
    page.get_pixmap(dpi=200).save(f"{OUT}/png200/T{i:02d}.png")
print(doc.page_count, "pages rendered")

## 2. PaddleOCR-VL 1.6 (one T4, PaddlePaddle engine; the vLLM route is not recommended on T4)

In [ ]:
!{UV} pip install --system -q paddlepaddle-gpu==3.2.1 --extra-index-url https://www.paddlepaddle.org.cn/packages/stable/cu126/ --index-strategy unsafe-best-match
!{UV} pip install --system -q "paddleocr[doc-parser]>=3.4.0"

In [ ]:
%%writefile /kaggle/working/paddle_run.py
import glob, json, os, time, traceback
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from paddleocr import PaddleOCRVL
OUT = "/kaggle/working/phase1"
dst = f"{OUT}/paddle"; os.makedirs(dst, exist_ok=True)
t0 = time.time(); pipe = PaddleOCRVL(pipeline_version="v1.6"); load_s = time.time() - t0
log = {"model": "PaddleOCR-VL v1.6", "load_seconds": round(load_s, 1), "pages": {}}
for png in sorted(glob.glob(f"{OUT}/png200/*.png")):
    name = os.path.basename(png)[:-4]
    t = time.time()
    try:
        for res in pipe.predict(png):
            res.save_to_markdown(save_path=f"{dst}/{name}")
            res.save_to_json(save_path=f"{dst}/{name}")
        md = "\n".join(open(f).read() for f in sorted(glob.glob(f"{dst}/{name}/*.md")))
        open(f"{dst}/{name}.md", "w").write(md)
        log["pages"][name] = {"seconds": round(time.time() - t, 1), "chars": len(md)}
    except Exception as e:
        log["pages"][name] = {"error": repr(e), "trace": traceback.format_exc()[-2000:]}
    print(name, log["pages"][name] if "error" in log["pages"][name] else log["pages"][name]["seconds"], flush=True)
json.dump(log, open(f"{dst}/log.json", "w"), indent=1)

In [ ]:
!python /kaggle/working/paddle_run.py 2>&1 | grep -v -i warning | tail -40

## 3. olmOCR-2 7B (split across both T4s in float16; T4s can't run its default bf16 format)

If this step fails or the output is empty or garbled, that's a result too: it tells us olmOCR-2 doesn't fit T4s.

In [ ]:
!{UV} pip install --system -q "transformers>=4.51" accelerate pyyaml
!{UV} pip install --system -q --no-deps "olmocr>=0.4.0"
!python -c "from olmocr.prompts import build_no_anchoring_v4_yaml_prompt as p; print(p()[:300])" || {UV} pip install --system -q "olmocr>=0.4.0"

In [ ]:
%%writefile /kaggle/working/olmo_run.py
import base64, glob, io, json, os, time, traceback
import pymupdf, torch
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from olmocr.prompts import build_no_anchoring_v4_yaml_prompt
OUT = "/kaggle/working/phase1"
PDF = glob.glob("/kaggle/input/**/phase1_pages.pdf", recursive=True)[0]
dst = f"{OUT}/olmocr"; os.makedirs(dst, exist_ok=True)
t0 = time.time()
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "allenai/olmOCR-2-7B-1025", torch_dtype=torch.float16, device_map="auto").eval()
proc = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
log = {"model": "olmOCR-2-7B-1025 fp16 device_map=auto", "load_seconds": round(time.time() - t0, 1), "pages": {}}
prompt = build_no_anchoring_v4_yaml_prompt()
doc = pymupdf.open(PDF)
for i, page in enumerate(doc, start=1):
    name = f"T{i:02d}"; t = time.time()
    try:
        zoom = 1288 / max(page.rect.width, page.rect.height)       # longest side 1288 px, as the model card says
        png = page.get_pixmap(matrix=pymupdf.Matrix(zoom, zoom)).tobytes("png")
        b64 = base64.b64encode(png).decode()
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}}]}]
        text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = proc(text=[text], images=[Image.open(io.BytesIO(png))], padding=True, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.inference_mode():
            out = model.generate(**inputs, temperature=0.1, do_sample=True, max_new_tokens=4096)
        new = out[:, inputs["input_ids"].shape[1]:]
        md = proc.tokenizer.batch_decode(new, skip_special_tokens=True)[0]
        open(f"{dst}/{name}.md", "w").write(md)
        log["pages"][name] = {"seconds": round(time.time() - t, 1), "chars": len(md), "new_tokens": int(new.shape[1])}
    except Exception as e:
        log["pages"][name] = {"error": repr(e), "trace": traceback.format_exc()[-2000:]}
    print(name, log["pages"][name] if "error" in log["pages"][name] else log["pages"][name]["seconds"], flush=True)
    json.dump(log, open(f"{dst}/log.json", "w"), indent=1)

In [ ]:
!python /kaggle/working/olmo_run.py 2>&1 | grep -v -i warning | tail -40

## 4. Pack the results

In [ ]:
import shutil
!pip list 2>/dev/null | grep -i -E "^(paddleocr|paddlepaddle-gpu|transformers|torch|olmocr) " > /kaggle/working/phase1/versions.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv >> /kaggle/working/phase1/versions.txt
shutil.rmtree("/kaggle/working/phase1/png200", ignore_errors=True)   # pictures are not needed back
shutil.make_archive("/kaggle/working/phase1_results", "zip", "/kaggle/working/phase1")
for m in ("paddle", "olmocr"):
    p = f"/kaggle/working/phase1/{m}/log.json"
    if os.path.exists(p):
        L = json.load(open(p)); ok = [v for v in L["pages"].values() if "seconds" in v]
        print(m, f"{len(ok)}/12 pages ok", f"avg {sum(v['seconds'] for v in ok)/max(len(ok),1):.1f} s/page")
    else:
        print(m, "no log (step failed)")
print("download: Output tab -> phase1_results.zip")